# Steam API

# Análisis de la API de Steam: Endpoints y Estructura de Datos

Para cumplir con el requisito de leer datos "LIVE" de una fuente pública, este proyecto consume información del ecosistema de Steam. Steam divide su acceso a datos en dos grandes bloques principales:

1. **Steam Store API (`appdetails`):** Es pública, no requiere API Key y está orientada a obtener metadatos del catálogo (precios, géneros, desarrolladores).
2. **Steam Web API:** Requiere una API Key privada y se utiliza para extraer métricas dinámicas de la comunidad, como jugadores en tiempo real (`GetNumberOfCurrentPlayers`) y el feed de noticias (`GetNewsForApp`).

A continuación, detallamos la estructura de los archivos JSON que devuelve cada endpoint.



In [2]:
# Importamos las librerías necesarias
import requests
import pandas as pd
import os
from dotenv import load_dotenv
import requests
import pandas as pd
import os
from dotenv import load_dotenv

In [3]:
# 1. Cargar variables de entorno desde el archivo .env
load_dotenv()

# 2. Guardar la API Key en una variable
STEAM_API_KEY = os.getenv("STEAM_API_KEY")
APP_ID = 730 # Counter-Strike 2

# Verificamos que la clave se ha cargado (sin imprimirla completa por seguridad)
if STEAM_API_KEY:
    print(f"API Key cargada correctamente (Empieza por: {STEAM_API_KEY[:4]}...)")
else:
    print("Error: No se ha encontrado la STEAM_API_KEY en el archivo .env")

API Key cargada correctamente (Empieza por: 643F...)


In [24]:
import requests
import pandas as pd
import os
import time
from dotenv import load_dotenv

load_dotenv()
STEAM_API_KEY = os.getenv("STEAM_API_KEY")

print("1. Obteniendo el Top 100 de juegos más jugados ahora mismo...")
url_top = f"https://api.steampowered.com/ISteamChartsService/GetGamesByConcurrentPlayers/v1/?key={STEAM_API_KEY}"
res_top = requests.get(url_top).json()
top_100 = res_top.get('response', {}).get('ranks', [])

# Convertimos a DataFrame inicial (solo tiene appid y concurrent_in_game)
df_jugadores = pd.DataFrame(top_100)
# Renombramos para que sea más claro
df_jugadores.rename(columns={'concurrent_in_game': 'jugadores_actuales'}, inplace=True)

appids_top_100 = df_jugadores['appid'].tolist()
datos_completos_tienda = []

print("2. Extrayendo la máxima información posible de la tienda para esos 100 juegos...")
print("Esto tardará aproximadamente 2 minutos para no saturar la API de Steam.")

for i, appid in enumerate(appids_top_100):
    url_store = f"https://store.steampowered.com/api/appdetails?appids={appid}&cc=es"
    res_store = requests.get(url_store).json()
    
    if res_store and str(appid) in res_store and res_store[str(appid)].get('success'):
        data = res_store[str(appid)]['data']
        
        # Extraer todo lo posible
        datos_completos_tienda.append({
            'appid': appid,
            'nombre': data.get('name', 'Desconocido'),
            'tipo': data.get('type', 'Desconocido'),
            'es_gratis': data.get('is_free', False),
            'precio_eur': data.get('price_overview', {}).get('final', 0) / 100 if not data.get('is_free', False) else 0.0,
            'metacritic_nota': data.get('metacritic', {}).get('score', 0),
            'desarrollador': data.get('developers', ['Desconocido'])[0],
            'editor': data.get('publishers', ['Desconocido'])[0],
            'windows': data.get('platforms', {}).get('windows', False),
            'mac': data.get('platforms', {}).get('mac', False),
            'linux': data.get('platforms', {}).get('linux', False),
            'fecha_lanzamiento': data.get('release_date', {}).get('date', 'Desconocida'),
            'edad_requerida': data.get('required_age', 0),
            # Unimos todos los géneros en un solo string separado por comas
            'generos': ", ".join([g['description'] for g in data.get('genres', [])])
        })
    
    # Progreso visual en consola
    if (i + 1) % 10 == 0:
        print(f"   -> Extraídos {i + 1}/100 juegos...")
        
    # LA REGLA DE ORO: Esperar 1.5 segundos para evitar el baneo de IP
    time.sleep(1.5)

# Convertir la lista gigante a DataFrame
df_tienda = pd.DataFrame(datos_completos_tienda)

print("\n3. Unificando (Merge) los datos en el Super DataFrame...")
# Juntamos la info de la tienda con la cantidad de jugadores usando el appid 
df_super = pd.merge(df_tienda, df_jugadores, on='appid', how='inner')

# Limpieza final: Ordenamos y reseteamos el índice 
df_super = df_super.sort_values(by='jugadores_actuales', ascending=False).reset_index(drop=True)

print("\n¡Proceso completado! Tienes un DataFrame con muchísimas dimensiones para analizar.")
display(df_super.head())

1. Obteniendo el Top 100 de juegos más jugados ahora mismo...
2. Extrayendo la máxima información posible de la tienda para esos 100 juegos...
Esto tardará aproximadamente 2 minutos para no saturar la API de Steam.
   -> Extraídos 10/100 juegos...
   -> Extraídos 20/100 juegos...
   -> Extraídos 30/100 juegos...
   -> Extraídos 40/100 juegos...
   -> Extraídos 50/100 juegos...
   -> Extraídos 60/100 juegos...
   -> Extraídos 70/100 juegos...
   -> Extraídos 80/100 juegos...
   -> Extraídos 90/100 juegos...
   -> Extraídos 100/100 juegos...

3. Unificando (Merge) los datos en el Super DataFrame...

¡Proceso completado! Tienes un DataFrame con muchísimas dimensiones para analizar.


,appid,nombre,tipo,es_gratis,precio_eur,metacritic_nota,desarrollador,editor,windows,mac,linux,fecha_lanzamiento,edad_requerida,generos,rank,jugadores_actuales,peak_in_game
0,730,Counter-Strike 2,game,True,0.00,0,Valve,Valve,True,False,True,"21 Aug, 2012",0,"Action, Free To Play",1,1432052,1510510
1,570,Dota 2,game,True,0.00,90,Valve,Valve,True,True,True,"9 Jul, 2013",0,"Action, Strategy, Free To Play",2,751649,781701
2,578080,PUBG: BATTLEGROUNDS,game,True,0.00,0,PUBG Corporation,"KRAFTON, Inc.",True,False,False,"21 Dec, 2017",16,"Action, Adventure, Massively Multiplayer, Free...",3,346784,704565
3,2868840,Slay the Spire 2,game,False,22.99,0,Mega Crit,Mega Crit,True,True,True,"5 Mar, 2026",0,"Indie, Strategy, Early Access",4,307260,430456
4,252490,Rust,game,False,39.99,69,Facepunch Studios,Facepunch Studios,True,True,False,"8 Feb, 2018",0,"Action, Adventure, Indie, Massively Multiplaye...",5,196377,196377


In [ ]:
df_super

## 1. Ficha Técnica y Metadatos (Steam Store API)
**Endpoint:** `https://store.steampowered.com/api/appdetails?appids={appid}`

Este endpoint devuelve un archivo JSON complejo con toda la información comercial del videojuego.

###  Diccionario de Variables (Store API)
* **`success`**: Booleano (`True`/`False`). Indica si el juego existe en la base de datos y la consulta fue exitosa.
* **`data`**: Diccionario principal que contiene toda la información del juego.
    * **`name`**: String. El título oficial del videojuego.
    * **`is_free`**: Booleano. `True` si es un título *Free-to-Play*, `False` si requiere compra.
    * **`price_overview`**: Diccionario con datos económicos (solo existe si `is_free` es `False`).
        * **`initial`**: Entero. Precio base sin descuentos, expresado en céntimos (ej. 1499 equivale a 14,99€).
        * **`final`**: Entero. Precio actual tras aplicar posibles descuentos, en céntimos.
    * **`developers` / `publishers`**: Listas de strings con los nombres de los estudios creadores y las empresas editoras.
    * **`genres`**: Lista de diccionarios. Detalla las categorías del juego (cada elemento incluye una `description` como "Acción", "Estrategia" o "Indie").
    * **`metacritic`**: Diccionario que incluye el `score` (entero de 0 a 100) otorgado por la crítica especializada.

In [2]:
# 1. Configurar y hacer la llamada HTTP
url_store = f"https://store.steampowered.com/api/appdetails?appids={APP_ID}&cc=es"
respuesta_store = requests.get(url_store).json()

# 2. Verificar que la petición fue exitosa y extraer el DataFrame
if respuesta_store and str(APP_ID) in respuesta_store and respuesta_store[str(APP_ID)].get('success'):
    
    # Navegamos directamente al diccionario "data" que contiene la información útil
    datos_completos_tienda = respuesta_store[str(APP_ID)]['data']
    
    # Convertimos TODO el JSON aplanado en un DataFrame
    df_store = pd.json_normalize(datos_completos_tienda)
    
    print(f"DataFrame 'Store' creado con éxito. Dimensiones: {df_store.shape}")
    display(df_store.head())
else:
    print("Error al obtener los datos de la tienda.")

DataFrame 'Store' creado con éxito. Dimensiones: (1, 52)


,type,name,steam_appid,required_age,is_free,dlc,detailed_description,about_the_game,short_description,supported_languages,...,ratings.agcom.descriptors,ratings.cadpa.rating,ratings.dejus.rating,ratings.dejus.descriptors,ratings.steam_germany.rating_generated,ratings.steam_germany.rating,ratings.steam_germany.required_age,ratings.steam_germany.banned,ratings.steam_germany.use_age_gate,ratings.steam_germany.descriptors
0,game,Counter-Strike 2,730,0,True,[2678630],"For over two decades, Counter-Strike has offer...","For over two decades, Counter-Strike has offer...","For over two decades, Counter-Strike has offer...","Czech, Danish, Dutch, English<strong>*</strong...",...,Violenza,12,16,Violência,1,16,16,0,0,Drastische Gewalt


## GRAFICAS

## 2. Métricas en Tiempo Real (Steam Web API)
**Endpoint:** `https://api.steampowered.com/ISteamUserStats/GetNumberOfCurrentPlayers/v1/?appid={appid}`

Un endpoint directo para medir el pulso actual de la comunidad. Devuelve un JSON muy ligero.

### 
 Diccionario de Variables (Jugadores Concurrentes)
* **`response`**: Diccionario contenedor principal de la respuesta.
    * **`player_count`**: Entero. Número exacto de cuentas de Steam a nivel mundial que tienen el juego ejecutándose en el mismo instante de realizar la petición HTTP.
    * **`result`**: Entero. Código de estado del servidor de Steam. Un valor de `1` indica que la petición fue procesada correctamente.

In [12]:
# 1. Configurar los parámetros y la URL
url_jugadores = "https://api.steampowered.com/ISteamUserStats/GetNumberOfCurrentPlayers/v1/"
parametros_jugadores = {
    "key": STEAM_API_KEY,
    "appid": APP_ID
}

# 2. Hacer la llamada HTTP
respuesta_jugadores = requests.get(url_jugadores, params=parametros_jugadores).json()

# 3. Convertir a DataFrame
if 'response' in respuesta_jugadores and respuesta_jugadores['response'].get('result') == 1:
    
    # Extraemos el diccionario 'response'
    datos_jugadores = respuesta_jugadores['response']
    
    df_jugadores = pd.json_normalize(datos_jugadores)
    
    # Añadimos la columna del AppID para no perder la referencia de a qué juego pertenece
    df_jugadores['appid'] = APP_ID 
    
    print(f"DataFrame 'Jugadores' creado con éxito. Dimensiones: {df_jugadores.shape}")
    display(df_jugadores)
else:
    print("Error al obtener los jugadores concurrentes.")

DataFrame 'Jugadores' creado con éxito. Dimensiones: (1, 3)


,player_count,result,appid
0,1422455,1,730


In [4]:
# 1. Configurar los parámetros y la URL
url_jugadores = "https://api.steampowered.com/ISteamUserStats/GetNumberOfCurrentPlayers/v1/"
parametros_jugadores = {
    "key": STEAM_API_KEY,
    "appid": APP_ID
}

# 2. Hacer la llamada HTTP
respuesta_jugadores = requests.get(url_jugadores, params=parametros_jugadores).json()

# 3. Convertir a DataFrame
if 'response' in respuesta_jugadores and respuesta_jugadores['response'].get('result') == 1:
    
    # Extraemos el diccionario 'response'
    datos_jugadores = respuesta_jugadores['response']
    
    df_jugadores = pd.json_normalize(datos_jugadores)
    
    # Añadimos la columna del AppID para no perder la referencia de a qué juego pertenece
    df_jugadores['appid'] = APP_ID 
    
    print(f"DataFrame 'Jugadores' creado con éxito. Dimensiones: {df_jugadores.shape}")
    display(df_jugadores)
else:
    print("Error al obtener los jugadores concurrentes.")

DataFrame 'Jugadores' creado con éxito. Dimensiones: (1, 3)


,player_count,result,appid
0,1118858,1,730


## GRAFICAS

## 3. Comunidad y Feed de Novedades (Steam Web API)
**Endpoint:** `http://api.steampowered.com/ISteamNews/GetNewsForApp/v0002/?appid={appid}`

Devuelve el historial de publicaciones, notas del parche y noticias relacionadas con el título, ideal para análisis de texto.

### Diccionario de Variables (Noticias)
* **`appnews`**: Diccionario contenedor principal.
    * **`appid`**: Entero. El identificador único del juego al que pertenecen las noticias.
    * **`newsitems`**: Lista de diccionarios. Cada diccionario representa una publicación individual.
        * **`title`**: String. El titular descriptivo de la noticia.
        * **`author`**: String. El autor o entidad que realizó la publicación (frecuentemente el estudio de desarrollo).
        * **`contents`**: String. El cuerpo de texto de la noticia. Puede contener formato HTML o estar truncado dependiendo de los parámetros de la llamada.
        * **`date`**: Entero. Fecha de publicación en formato *Unix Timestamp* (segundos transcurridos desde el 1 de enero de 1970). Requiere conversión para su lectura humana.

In [8]:
# 1. Configurar los parámetros y la URL
url_noticias = "http://api.steampowered.com/ISteamNews/GetNewsForApp/v0002/"
parametros_noticias = {
    "key": STEAM_API_KEY,
    "appid": APP_ID,
    "count": 10,       # Número de noticias a extraer
    "maxlength": 300,  # Longitud máxima del texto de la noticia
    "format": "json"
}

# 2. Hacer la llamada HTTP
respuesta_noticias = requests.get(url_noticias, params=parametros_noticias).json()

# 3. Convertir a DataFrame
if 'appnews' in respuesta_noticias and 'newsitems' in respuesta_noticias['appnews']:
    
    # Extraemos la LISTA de diccionarios de noticias
    lista_noticias = respuesta_noticias['appnews']['newsitems']
    
    # Al pasar una lista, json_normalize crea una fila por cada elemento
    df_noticias = pd.json_normalize(lista_noticias)
    
    print(f"DataFrame 'Noticias' creado con éxito. Dimensiones: {df_noticias.shape}")
    display(df_noticias.head())
else:
    print("Error al obtener las noticias.")

DataFrame 'Noticias' creado con éxito. Dimensiones: (10, 12)


,gid,title,url,is_external_url,author,contents,feedlabel,date,feedname,feed_type,appid,tags
0,1826362059919569,Counter-Strike 2 Update,https://steamstore-a.akamaihd.net/news/externa...,True,Vitaliy,"\Starting today, items listed for sale on Stea...",Community Announcements,1772664700,steam_community_announcements,1,730,[patchnotes]
1,1826362059917935,"CSGO is back on Steam, if you know where to look",https://steamstore-a.akamaihd.net/news/externa...,True,editor@pcgamesn.com,<strong>Counter-Strike 2</strong> has largely ...,PCGamesN,1772623159,PCGamesN,0,730,NaN
2,1825727806706141,Counter-Strike 2 Update,https://steamstore-a.akamaihd.net/news/externa...,True,jo,\Added Instance.SetSaveDataAdded Instance.GetS...,Community Announcements,1772060032,steam_community_announcements,1,730,[patchnotes]
3,1825093633195926,Counter-Strike 2 Update,https://steamstore-a.akamaihd.net/news/externa...,True,jo,\Mitigated a performance issue that primarily ...,Community Announcements,1771889921,steam_community_announcements,1,730,[patchnotes]
4,1823825466506792,Counter-Strike 2 Update,https://steamstore-a.akamaihd.net/news/externa...,True,jo,\Localization code and text changes.,Community Announcements,1770677957,steam_community_announcements,1,730,[patchnotes]


## GRAFICAS